# Limpeza e validação dos dados

Este notebook documenta a limpeza do dataset de estatísticas de jogadores da Issue #7. A análise utiliza o arquivo original e as funções reutilizáveis de `src/preprocessing/data_cleaning.py`.

## 1. Objetivo da limpeza

O objetivo é verificar a qualidade estrutural e semântica dos dados, preservar informações válidas, manter valores ausentes quando eles representam ausência de informação e converter tipos que precisam ser usados pelo processamento posterior. A limpeza não preenche nulos automaticamente, não remove outliers sem justificativa e não elimina registros válidos.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

project_root = Path.cwd()
if not (project_root / "data" / "raw").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.preprocessing.data_cleaning import (
    DATE_COLUMNS,
    INTEGER_COLUMNS,
    NUMERIC_COLUMNS,
    clean_dataset,
    detect_inconsistencies,
    identify_duplicates,
    identify_missing_values,
    validate_cleaned_dataset,
)

raw_path = project_root / "data" / "raw" / "tb_lobby_stats_player.csv"
df_raw = pd.read_csv(raw_path)

## 2. Apresentação do dataset original

In [2]:
dataset_summary = pd.DataFrame({
    "métrica": ["Linhas", "Colunas", "Jogadores únicos", "Lobbies únicos"],
    "valor": [
        len(df_raw),
        len(df_raw.columns),
        df_raw["idPlayer"].nunique(),
        df_raw["idLobbyGame"].nunique(),
    ],
})
display(dataset_summary)
display(df_raw.head())

,métrica,valor
0,Linhas,184152
1,Colunas,38
2,Jogadores únicos,2469
3,Lobbies únicos,172911


,idLobbyGame,idPlayer,idRoom,qtKill,qtAssist,qtDeath,qtHs,qtBombeDefuse,qtBombePlant,qtTk,qtTkAssist,qt1Kill,qt2Kill,qt3Kill,qt4Kill,qt5Kill,qtPlusKill,qtFirstKill,vlDamage,qtHits,qtShots,qtLastAlive,qtClutchWon,qtRoundsPlayed,descMapName,vlLevel,qtSurvived,qtTrade,qtFlashAssist,qtHitHeadshot,qtHitChest,qtHitStomach,qtHitLeftAtm,qtHitRightArm,qtHitLeftLeg,qtHitRightLeg,flWinner,dtCreatedAt
0,1,1,1,5,1,16,2,0,0,0.0,0.0,3,1,0,0,0,0,1,911,25.0,212,0.0,0,17,de_mirage,10,1.0,2.0,0.0,3.0,13.0,4.0,2.0,2.0,1.0,0.0,0,2022-01-21 19:45:44
1,2,1,2,24,3,18,6,0,4,0.0,1.0,9,4,1,1,0,0,3,2933,54.0,597,1.0,1,27,de_nuke,9,8.0,1.0,0.0,7.0,26.0,14.0,2.0,1.0,1.0,3.0,1,2022-02-04 02:09:47
2,3,2,3,6,4,23,2,0,1,0.0,1.0,4,1,0,0,0,0,2,1117,31.0,207,0.0,0,25,de_mirage,2,2.0,3.0,0.0,3.0,15.0,8.0,1.0,2.0,0.0,2.0,0,2021-09-18 18:07:43
3,3,391,27508,10,5,20,4,1,0,0.0,0.0,6,2,0,0,0,0,1,1740,63.0,411,1.0,1,25,de_mirage,15,3.0,4.0,0.0,6.0,27.0,10.0,1.0,7.0,6.0,6.0,1,2021-09-18 18:07:43
4,4,2,4,8,4,26,6,0,2,0.0,0.0,4,2,0,0,0,0,1,1696,51.0,324,0.0,0,30,de_nuke,2,4.0,4.0,2.0,8.0,19.0,12.0,2.0,3.0,2.0,5.0,0,2021-09-27 00:17:45


## 3. Diagnóstico de valores ausentes

Os valores ausentes são tratados como ausência de informação. O diagnóstico abaixo mostra a quantidade por coluna, a quantidade de linhas afetadas e os padrões de ausência encontrados.

In [3]:
missing_report = identify_missing_values(df_raw)
missing_report = missing_report.sort_values(["missing_count", "column"], ascending=[False, True])
rows_with_missing = int(df_raw.isna().any(axis=1).sum())
missing_columns = missing_report[missing_report["missing_count"] > 0]
pattern_series = (
    df_raw.loc[df_raw.isna().any(axis=1)]
    .isna()
    .apply(lambda row: tuple(row.index[row]), axis=1)
    .value_counts()
    .rename_axis("colunas_ausentes")
    .reset_index(name="linhas")
)
display(missing_columns)
display(pd.DataFrame({"métrica": ["Linhas com algum valor ausente", "Valores ausentes"], "valor": [rows_with_missing, int(df_raw.isna().sum().sum())]}))
display(pattern_series)

,column,missing_count,missing_rate
28,qtFlashAssist,705,0.003828
30,qtHitChest,705,0.003828
29,qtHitHeadshot,705,0.003828
32,qtHitLeftAtm,705,0.003828
34,qtHitLeftLeg,705,0.003828
33,qtHitRightArm,705,0.003828
35,qtHitRightLeg,705,0.003828
31,qtHitStomach,705,0.003828
26,qtSurvived,705,0.003828
27,qtTrade,705,0.003828


,métrica,valor
0,Linhas com algum valor ausente,705
1,Valores ausentes,7530


,colunas_ausentes,linhas
0,"(qtSurvived, qtTrade, qtFlashAssist, qtHitHead...",585
1,"(qtTk, qtTkAssist, qtHits, qtLastAlive, qtSurv...",120


## 4. Diagnóstico de duplicatas

In [4]:
complete_duplicates = identify_duplicates(df_raw)
player_lobby_duplicates = identify_duplicates(df_raw, subset=["idPlayer", "idLobbyGame"])
duplicate_summary = pd.DataFrame({
    "verificação": [
        "Duplicatas completas",
        "Duplicatas em idPlayer + idLobbyGame",
    ],
    "linhas_envolvidas": [len(complete_duplicates), len(player_lobby_duplicates)],
})
display(duplicate_summary)

,verificação,linhas_envolvidas
0,Duplicatas completas,0
1,Duplicatas em idPlayer + idLobbyGame,0


## 5. Verificação dos tipos de dados

Os identificadores e contagens devem ser inteiros anuláveis, `vlDamage` deve ser numérico e `dtCreatedAt` deve ser convertido para datetime. A tabela compara o tipo carregado do arquivo com o tipo esperado pelo módulo de limpeza.

In [5]:
type_rows = []
for column in df_raw.columns:
    if column in INTEGER_COLUMNS:
        expected_type = "Int64"
    elif column in NUMERIC_COLUMNS:
        expected_type = "numérico"
    elif column in DATE_COLUMNS:
        expected_type = "datetime"
    else:
        expected_type = "texto ou categórico"
    type_rows.append({"coluna": column, "tipo_atual": str(df_raw[column].dtype), "tipo_esperado": expected_type})
type_report = pd.DataFrame(type_rows)
display(type_report)

,coluna,tipo_atual,tipo_esperado
0,idLobbyGame,int64,Int64
1,idPlayer,int64,Int64
2,idRoom,int64,Int64
3,qtKill,int64,Int64
4,qtAssist,int64,Int64
5,qtDeath,int64,Int64
6,qtHs,int64,Int64
7,qtBombeDefuse,int64,Int64
8,qtBombePlant,int64,Int64
9,qtTk,float64,Int64


## 6. Investigação de valores inconsistentes

O dicionário de dados registrado no notebook `00_compreensao_dataset.ipynb` define `qtKill` como a quantidade total de kills, `qtHs` como a quantidade de kills por headshot, `qtDeath` como a quantidade de mortes e `qtRoundsPlayed` como a quantidade de rounds disputados pelo jogador.

Assim, `qtHs > qtKill` contradiz diretamente a interpretação de headshots como parte das kills totais. No CSV, isso ocorre em 2 registros. Já `qtDeath > qtRoundsPlayed` contradiz a expectativa usual de no máximo uma morte por round jogado; isso ocorre em 414 registros, distribuídos por 413 lobbies e 317 jogadores. O arquivo não informa se a origem usa uma regra de contabilização diferente, se houve desconexão ou se houve falha de telemetria. Por isso, esses casos são inconsistências semânticas confirmadas pela relação numérica, mas não são prova suficiente para corrigir os valores ou excluir os registros.

A função de validação classifica essas relações em `review_flags`, e não em `blocking_issues`. Essa classificação permite comunicar a ocorrência sem tratar valores ausentes legítimos como erro e sem remover dados válidos automaticamente.

In [6]:
inconsistencies = detect_inconsistencies(df_raw)
category_labels = {
    "negative_values": "Valores negativos",
    "invalid_numeric_values": "Valores numéricos inválidos",
    "invalid_dates": "Datas inválidas",
    "invalid_winner": "Valores inválidos em flWinner",
    "duplicate_rows": "Linhas duplicadas",
    "duplicate_player_lobby": "Duplicidades em idPlayer + idLobbyGame",
    "headshots_exceed_kills": "Headshots acima de kills",
    "deaths_exceed_rounds": "Mortes acima de rounds",
    "hits_exceed_shots": "Hits acima de tiros",
    "headshot_hits_exceed_hits": "Headshots acima de hits",
    "kills_exceed_shots": "Kills acima de tiros",
}
inconsistency_rows = []
for category, value in inconsistencies.items():
    label = category_labels.get(category, category)
    if isinstance(value, dict):
        if value:
            for column, count in value.items():
                inconsistency_rows.append({"verificação": label, "detalhe": column, "linhas": count})
        else:
            inconsistency_rows.append({"verificação": label, "detalhe": "nenhuma", "linhas": 0})
    else:
        inconsistency_rows.append({"verificação": label, "detalhe": "geral", "linhas": value})
body_hit_columns = ["qtHitHeadshot", "qtHitChest", "qtHitStomach", "qtHitLeftAtm", "qtHitRightArm", "qtHitLeftLeg", "qtHitRightLeg"]
body_complete = df_raw[body_hit_columns + ["qtHits"]].notna().all(axis=1)
body_hit_sum = df_raw.loc[body_complete, body_hit_columns].sum(axis=1)
body_hit_mismatch = int((body_hit_sum != df_raw.loc[body_complete, "qtHits"]).sum())
inconsistency_rows.append({"verificação": "Soma das regiões do corpo diferente de qtHits", "detalhe": "geral", "linhas": body_hit_mismatch})
display(pd.DataFrame(inconsistency_rows))
relation_columns = ["idLobbyGame", "idPlayer", "qtRoundsPlayed", "qtKill", "qtHs", "qtDeath", "qtHitHeadshot", "qtHits", "qtShots"]
headshot_rows = df_raw.loc[df_raw["qtHs"] > df_raw["qtKill"], relation_columns]
death_rows = df_raw.loc[df_raw["qtDeath"] > df_raw["qtRoundsPlayed"], relation_columns]
death_difference = death_rows["qtDeath"] - death_rows["qtRoundsPlayed"]
relation_summary = pd.DataFrame({"verificação": ["qtHs > qtKill", "qtDeath > qtRoundsPlayed"], "linhas": [len(headshot_rows), len(death_rows)], "lobbies": [headshot_rows["idLobbyGame"].nunique(), death_rows["idLobbyGame"].nunique()], "jogadores": [headshot_rows["idPlayer"].nunique(), death_rows["idPlayer"].nunique()], "maior_diferença": [int((headshot_rows["qtHs"] - headshot_rows["qtKill"]).max()), int(death_difference.max())]})
display(headshot_rows)
display(relation_summary)
display(death_difference.value_counts().sort_index().rename_axis("diferença").reset_index(name="linhas"))

,verificação,detalhe,linhas
0,Valores negativos,nenhuma,0
1,Valores numéricos inválidos,nenhuma,0
2,Datas inválidas,nenhuma,0
3,Valores inválidos em flWinner,geral,0
4,Linhas duplicadas,geral,0
5,Duplicidades em idPlayer + idLobbyGame,geral,0
6,Headshots acima de kills,geral,2
7,Mortes acima de rounds,geral,414
8,Hits acima de tiros,geral,0
9,Headshots acima de hits,geral,0


,idLobbyGame,idPlayer,qtRoundsPlayed,qtKill,qtHs,qtDeath,qtHitHeadshot,qtHits,qtShots
58734,52799,777,5,0,2,3,2.0,4.0,58
162076,150969,2310,24,0,7,17,9.0,46.0,343


,verificação,linhas,lobbies,jogadores,maior_diferença
0,qtHs > qtKill,2,2,2,7
1,qtDeath > qtRoundsPlayed,414,413,317,13


,diferença,linhas
0,1,240
1,2,92
2,3,34
3,4,25
4,5,11
5,6,5
6,7,3
7,8,2
8,11,1
9,13,1


## 7. Análise de valores extremos relevantes

Valores extremos são apresentados para investigação. Eles não são removidos automaticamente, pois podem representar partidas legítimas e a análise exploratória não fornece justificativa para descartá-los.

In [7]:
extreme_columns = ["qtKill", "vlDamage", "qtShots", "qtHits", "qtRoundsPlayed", "qtHitHeadshot", "qtDeath"]
extreme_rows = []
for column in extreme_columns:
    row_index = df_raw[column].idxmax()
    row = df_raw.loc[row_index]
    extreme_rows.append({
        "coluna": column,
        "p99": df_raw[column].quantile(0.99),
        "p99_9": df_raw[column].quantile(0.999),
        "máximo": row[column],
        "idLobbyGame": row["idLobbyGame"],
        "idPlayer": row["idPlayer"],
    })
extreme_report = pd.DataFrame(extreme_rows)
display(extreme_report)
display(pd.DataFrame({"verificação": ["qtRoundsPlayed acima de 42", "qtRoundsPlayed acima de 30"], "linhas": [(df_raw["qtRoundsPlayed"] > 42).sum(), (df_raw["qtRoundsPlayed"] > 30).sum()]}))

,coluna,p99,p99_9,máximo,idLobbyGame,idPlayer
0,qtKill,39.00,50.000,85.0,117343,1864
1,vlDamage,4900.00,6180.698,10794.0,117343,1864
2,qtShots,966.49,1285.849,2131.0,72520,1138
3,qtHits,128.00,163.000,277.0,117343,1864
4,qtRoundsPlayed,42.00,54.000,84.0,117343,1863
5,qtHitHeadshot,23.00,30.000,49.0,75716,1181
6,qtDeath,33.00,41.000,65.0,117343,1863


,verificação,linhas
0,qtRoundsPlayed acima de 42,1742
1,qtRoundsPlayed acima de 30,19895


## 8. Justificativa dos tratamentos escolhidos

In [8]:
treatment_summary = pd.DataFrame([
    {"problema": "Valores ausentes", "tratamento": "Preservar os nulos", "justificativa": f"{rows_with_missing} linhas possuem ausência de informação; nulos não são equivalentes automaticamente a zero."},
    {"problema": "Duplicatas", "tratamento": "Não remover automaticamente", "justificativa": f"A investigação encontrou {len(complete_duplicates)} linhas duplicadas completas e {len(player_lobby_duplicates)} duplicidades na chave do jogador e lobby."},
    {"problema": "Tipos", "tratamento": "Converter tipos numéricos e datas", "justificativa": "A conversão é necessária para preservar os valores válidos e manter a compatibilidade com as features históricas."},
    {"problema": "Inconsistências semânticas", "tratamento": "Sinalizar para revisão", "justificativa": "Relações suspeitas não são corrigidas sem evidência da regra de negócio."},
    {"problema": "Valores extremos", "tratamento": "Preservar e investigar", "justificativa": "Um valor extremo pode ser uma partida legítima e não deve ser excluído automaticamente."},
])
display(treatment_summary)

,problema,tratamento,justificativa
0,Valores ausentes,Preservar os nulos,705 linhas possuem ausência de informação; nul...
1,Duplicatas,Não remover automaticamente,A investigação encontrou 0 linhas duplicadas c...
2,Tipos,Converter tipos numéricos e datas,A conversão é necessária para preservar os val...
3,Inconsistências semânticas,Sinalizar para revisão,Relações suspeitas não são corrigidas sem evid...
4,Valores extremos,Preservar e investigar,Um valor extremo pode ser uma partida legítima...


## 9. Execução da função `clean_dataset`

In [9]:
df_clean = clean_dataset(df_raw)
validation = validate_cleaned_dataset(df_clean)
validation_summary = pd.DataFrame({
    "métrica": ["Resultado estrutural", "Bloqueios críticos", "Sinais para revisão", "Linhas", "Colunas"],
    "valor": [
        "válido" if validation["is_valid"] else "requer correção",
        validation["blocking_issues"] or "nenhum",
        validation["review_flags"] or "nenhum",
        validation["rows"],
        validation["columns"],
    ],
})
display(validation_summary)
review_flag_report = pd.DataFrame([{"verificação": name, "linhas": count} for name, count in validation["review_flags"].items()])
display(review_flag_report)

,métrica,valor
0,Resultado estrutural,válido
1,Bloqueios críticos,nenhum
2,Sinais para revisão,"{'headshots_exceed_kills': 2, 'deaths_exceed_r..."
3,Linhas,184152
4,Colunas,38


,verificação,linhas
0,headshots_exceed_kills,2
1,deaths_exceed_rounds,414


## 10. Comparação antes e depois da limpeza

In [10]:
comparison = pd.DataFrame({
    "métrica": [
        "Linhas",
        "Colunas",
        "Linhas com algum valor ausente",
        "Valores ausentes",
        "Duplicatas completas",
        "Duplicatas em idPlayer + idLobbyGame",
    ],
    "antes": [
        len(df_raw),
        len(df_raw.columns),
        int(df_raw.isna().any(axis=1).sum()),
        int(df_raw.isna().sum().sum()),
        len(identify_duplicates(df_raw)),
        len(identify_duplicates(df_raw, subset=["idPlayer", "idLobbyGame"])),
    ],
    "depois": [
        len(df_clean),
        len(df_clean.columns),
        int(df_clean.isna().any(axis=1).sum()),
        int(df_clean.isna().sum().sum()),
        len(identify_duplicates(df_clean)),
        len(identify_duplicates(df_clean, subset=["idPlayer", "idLobbyGame"])),
    ],
})
display(comparison)
type_changes = pd.DataFrame([
    {"coluna": column, "antes": str(df_raw[column].dtype), "depois": str(df_clean[column].dtype)}
    for column in df_raw.columns
    if str(df_raw[column].dtype) != str(df_clean[column].dtype)
])
display(type_changes)

,métrica,antes,depois
0,Linhas,184152,184152
1,Colunas,38,38
2,Linhas com algum valor ausente,705,705
3,Valores ausentes,7530,7530
4,Duplicatas completas,0,0
5,Duplicatas em idPlayer + idLobbyGame,0,0


,coluna,antes,depois
0,idLobbyGame,int64,Int64
1,idPlayer,int64,Int64
2,idRoom,int64,Int64
3,qtKill,int64,Int64
4,qtAssist,int64,Int64
5,qtDeath,int64,Int64
6,qtHs,int64,Int64
7,qtBombeDefuse,int64,Int64
8,qtBombePlant,int64,Int64
9,qtTk,float64,Int64


## 11. Resumo das alterações aplicadas

In [11]:
applied_changes = pd.DataFrame([
    {"alteração": "Cópia do DataFrame", "resultado": "O dataset original não é modificado."},
    {"alteração": "Tipos numéricos", "resultado": f"{len(type_changes)} colunas tiveram o tipo ajustado conforme o esquema esperado."},
    {"alteração": "Datas", "resultado": "dtCreatedAt foi convertido para datetime."},
    {"alteração": "Valores ausentes", "resultado": f"{int(df_clean.isna().sum().sum())} valores ausentes foram preservados."},
    {"alteração": "Registros", "resultado": f"Nenhum registro foi removido; o total permaneceu em {len(df_clean)}."},
    {"alteração": "Outliers", "resultado": "Nenhum outlier foi removido automaticamente."},
])
display(applied_changes)

,alteração,resultado
0,Cópia do DataFrame,O dataset original não é modificado.
1,Tipos numéricos,36 colunas tiveram o tipo ajustado conforme o ...
2,Datas,dtCreatedAt foi convertido para datetime.
3,Valores ausentes,7530 valores ausentes foram preservados.
4,Registros,Nenhum registro foi removido; o total permanec...
5,Outliers,Nenhum outlier foi removido automaticamente.


## 12. Conclusões sobre a qualidade dos dados

In [12]:
review_text = "; ".join(f"{name}={count}" for name, count in validation["review_flags"].items()) or "nenhum"
conclusion = (
    f"O dataset possui {len(df_raw)} linhas e {len(df_raw.columns)} colunas. Foram identificadas {rows_with_missing} linhas com algum valor ausente, e esses valores foram preservados por representarem ausência de informação. "
    f"A validação estrutural não encontrou bloqueios críticos. Os sinais semânticos mantidos para investigação foram: {review_text}. "
    f"A limpeza não removeu registros ou valores extremos e manteve {len(df_clean)} linhas. O resultado é compatível com o processamento de histórico dos jogadores."
)
print(conclusion)

O dataset possui 184152 linhas e 38 colunas. Foram identificadas 705 linhas com algum valor ausente, e esses valores foram preservados por representarem ausência de informação. A validação estrutural não encontrou bloqueios críticos. Os sinais semânticos mantidos para investigação foram: headshots_exceed_kills=2; deaths_exceed_rounds=414. A limpeza não removeu registros ou valores extremos e manteve 184152 linhas. O resultado é compatível com o processamento de histórico dos jogadores.


## 13. Integração com as features históricas

A integração aplica `criar_features_historico` ao resultado validado da limpeza e depois aplica `filtrar_minimo_partidas`. A implementação ordena por jogador, data e lobby e usa somente partidas anteriores. As verificações abaixo registram as quantidades, as 25 features esperadas, a preservação das colunas originais e a permanência dos sinais de revisão.

In [13]:
from src.preprocessing.data_cleaning import detect_inconsistencies
from src.preprocessing.feature_historico import criar_features_historico, filtrar_minimo_partidas

df_features = criar_features_historico(df_clean)
df_eligible = filtrar_minimo_partidas(df_features)
source_columns = list(df_clean.columns)
historical_columns = [column for column in df_features.columns if column not in source_columns]
ordered_clean = df_clean.sort_values(["idPlayer", "dtCreatedAt", "idLobbyGame"]).reset_index(drop=True)
source_values_preserved = all(df_features[column].reset_index(drop=True).equals(ordered_clean[column]) for column in source_columns)
expected_previous = ordered_clean.groupby("idPlayer")["qtKill"].shift(1)
expected_count = ordered_clean.groupby("idPlayer").cumcount()
same_player_context = df_features["qtdAnteriores"].equals(expected_count) and df_features["killsUltimaPartida"].equals(expected_previous)
review_names = ["headshots_exceed_kills", "deaths_exceed_rounds"]
review_counts = pd.DataFrame([
    {"etapa": etapa, **{name: detect_inconsistencies(frame)[name] for name in review_names}}
    for etapa, frame in [("limpeza", df_clean), ("features", df_features), ("filtro", df_eligible)]
])
integration_summary = pd.DataFrame({
    "métrica": ["Linhas após limpeza", "Linhas após features", "Linhas elegíveis", "Features históricas criadas", "Menor qtdAnteriores após filtro", "Colunas originais preservadas", "Contexto do mesmo jogador preservado"],
    "valor": [len(df_clean), len(df_features), len(df_eligible), len(historical_columns), int(df_eligible["qtdAnteriores"].min()), source_values_preserved, same_player_context],
})
missing_summary = pd.DataFrame({
    "etapa": ["CSV original", "Dataset limpo", "Colunas originais após features", "Features históricas após filtro"],
    "valores_ausentes": [int(df_raw.isna().sum().sum()), int(df_clean.isna().sum().sum()), int(df_features[source_columns].isna().sum().sum()), int(df_eligible[historical_columns].isna().sum().sum())],
})
display(integration_summary)
display(missing_summary)
display(review_counts)

,métrica,valor
0,Linhas após limpeza,184152
1,Linhas após features,184152
2,Linhas elegíveis,172863
3,Features históricas criadas,25
4,Menor qtdAnteriores após filtro,5
5,Colunas originais preservadas,True
6,Contexto do mesmo jogador preservado,True


,etapa,valores_ausentes
0,CSV original,7530
1,Dataset limpo,7530
2,Colunas originais após features,7530
3,Features históricas após filtro,138


,etapa,headshots_exceed_kills,deaths_exceed_rounds
0,limpeza,2,414
1,features,2,414
2,filtro,1,363
